# Mission 2 — 신고자 / 119대원 화자 분류 (WavLM-base 12층 fine-tuning) 학습 기록

이 노트북은 최종 채택 모델(12층 전체 fine-tuning)의 학습 과정을 기록한 자료입니다. 채점 시 실행되는 파일은 `inference.py`입니다.

**학습 설정 요약 및 재현성 정보**
| 항목 | 내용 |
|---|---|
| 모델 | `microsoft/wavlm-base` + Attention Pooling + Dropout(0.3) + Linear(2) |
| 입력 | 16kHz, 2.5초 고정 길이 (짧으면 뒤를 0으로 패딩, 길면 앞부분 사용), 발화별 z-score 정규화 |
| 학습 데이터 | 대회 Training(서울) 29,200개 통화 중 300ms 이상 발화 880,178개 |
| Train/Val 분리 | 통화(wav) 단위 8:2, `random.seed(42)` (같은 통화의 발화가 양쪽에 섞이지 않음) |
| 최적화 | Adam(weight_decay=1e-4), 기준 lr 1e-5에 층이 아래로 갈수록 ×0.9 감쇠, ReduceLROnPlateau(Val Acc, ×0.5, patience 1) |
| 손실 / batch / epoch | CrossEntropy / 32 / 최대 15 (Early Stopping patience 4) |
| 체크포인트 선택 | 위 8:2 분할의 **자체 Val** Loss가 최저인 epoch의 가중치를 저장 |
| 환경 | 학교 서버, RTX 3080 10GB, i7-13700K, RAM 128GB |
| 난수 시드 | 데이터 분할만 `random.seed(42)`로 고정. 모델 초기화(Attention Pooling·분류기)와 DataLoader 셔플은 시드를 고정하지 않았으므로 동일한 수치의 완전 재현은 보장되지 않음 |

**Validation 사용 방식**
- 자체 Val(8:2 분할의 20%)은 순전파(`torch.no_grad`)로 성능 확인, 학습률 조정, 체크포인트 선택에만 사용했고 역전파에는 쓰지 않았습니다.
- 대회 공식 Validation(3,640통화)은 학습과 체크포인트 선택에 일절 사용하지 않았고, 맨 아래 14번 셀에서 최종 평가에만 사용합니다.


In [1]:
# 0. 라이브러리 및 디바이스 확인

import os # 파일 경로 처리
import json # json 라벨 파일 읽기
import glob # 파일 목록 검색
import random # train / val 분할
import numpy as np # 배열 연산
import torch # 딥러닝 프레임워크
import torch.nn as nn # 신경망 레이어 정의 용도
import torch.optim as optim # optimizer (Adam)
from torch.utils.data import Dataset, DataLoader # 데이터 파이프라인 구성
import librosa # 오디오 리샘플링
import soundfile as sf # wav 파일 읽기
from tqdm import tqdm # 진행률 표시
import time # 학습 시간 측정
from transformers import WavLMModel # WavLM 로드

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # GPU 없으면 CPU 사용
print("device:", device) # 어떤 디바이스로 돌아가는지 확인
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
# GPU 모델명 (학교 서버 사용 - 확인 용도)

device: cuda
GPU: NVIDIA GeForce RTX 3080


In [2]:
# 1. 경로 설정 (Traning 데이터 - wav 29200개)

label_dir = os.path.expanduser("~/mission2_wavlm/data/labels_local") # json 라벨 파일 경로
json_paths = glob.glob(os.path.join(label_dir, "*.json")) # 해당 폴더의 모든 json 파일 경로 목록

wav_dir_base = os.path.expanduser("~/mission2_wavlm/data/raw_audio") # wav 폴더 경로
all_wavs = glob.glob(os.path.join(wav_dir_base, "**", "*.wav"), recursive=True) # 하위 폴더까지 wav 전부 탐색
wav_path_map = {os.path.basename(p): p for p in all_wavs} # 파일명으로만 전체 경로를 찾기 위한 딕셔너리 사용
print("wav 개수:", len(wav_path_map)) # 실제 확보된 wav 개수

wav 개수: 29200


In [3]:
# 2. file_list 생성 - json 발화 구간 (시작/끝) 뽑기

file_list = [] # (wav_name, start_ms, end_ms, speaker)를 담을 리스트

for jp in tqdm(json_paths, desc="file_list 생성"): # 모든 json파일을 순회하며 진행률 표시
    try: # 하나의 문제로 전체가 멈추지 않게 예외 처리
        with open(jp, 'r', encoding='utf-8') as f: # json file open
            data = json.load(f) # json 내용을 파이썬 딕셔너리로 파싱

        # 처음에 AI-hub에서 직접 다운 받아서 작업을 시작해서 해당 로직이 추가되었습니다!
        wav_name = os.path.basename(jp).replace('.json', '.wav') # json 파일 명으로 wav 매칭
        if wav_name not in wav_path_map: # 없으면 해당 json 건너뛰기 
            continue

        for utt in data.get('utterances', []): # 통화 안의 모든 발화 순회
            start_ms = utt.get('startAt') # 발화 시작 시간
            end_ms = utt.get('endAt') # 발화 끝 시간
            speaker = utt.get('speaker') # 화자 라벨
            if start_ms is None or end_ms is None or end_ms <= start_ms or speaker is None:
                # 시간 및 화자 정보가 비정상적이면 건너뛰기
                continue
            duration = end_ms - start_ms # 발화 길이 계산 (ms)
            if duration < 300: # 300ms 미만이면 학습에서 제외
                continue
            file_list.append((wav_name, start_ms, end_ms, speaker)) # 조건을 통과한 발화를 리스트에 추가
    except Exception: # json 파싱 등에서 예외 시 건너뜀
        continue

print(f"총 샘플 수: {len(file_list)}") # 최종적으로 모인 발화 조각 개수

file_list 생성: 100%|█████████████████████████████████████████████████████████| 62362/62362 [00:04<00:00, 14324.25it/s]

총 샘플 수: 880178


In [4]:
# 4. Train/Val 분리 (wav 파일 단위)

random.seed(42) # 재현성을 위해 랜덤 시드 고정해서 사용
unique_wavs = sorted(set([f[0] for f in file_list])) # 중복 없는 wav 파일명 목록을 만들고 정렬
random.shuffle(unique_wavs) # 고정된 순서를 설정한 시드 기준으로 섞음

split_idx = int(len(unique_wavs) * 0.8) # 전체 80% 지점 인덱스 계산
train_wavs = set(unique_wavs[:split_idx]) # 앞 80%는 train
val_wavs = set(unique_wavs[split_idx:]) # 뒤 20%는 val

train_list = [f for f in file_list if f[0] in train_wavs] # train에 속한 발화만 모아서 list 생성
val_list = [f for f in file_list if f[0] in val_wavs] # val에 속한 발화만 모아서 list 생성

random.shuffle(train_list) # train 배치 구성 시 순서 편향이 없도록 섞음
random.shuffle(val_list) # 위와 동일

print(f"train: {len(train_list)}, val: {len(val_list)}") # 분할된 각각 발화 개수 확인

train: 703394, val: 176784


In [5]:
# 5. Dataset 정의 - 발화 구간을 모델 입력으로 변환 (16kHz, 2.5초 고정 길이 파형)

class WavLMDataset(Dataset): # PyTorch Dataset 상속
    def __init__(self, file_list, wav_path_map, sr=16000, max_duration=2.5):
        self.file_list = file_list # 발화 목록 저장
        self.wav_path_map = wav_path_map # 파일명 -> 경로 매핑 저장
        self.sr = sr # 목표 샘플레이트 저장
        self.max_len = int(sr * max_duration) # 목표 길이를 샘플 개수로 환산해 저장 (16000 * 2.5 = 40000)

    def __len__(self): # Dataset의 전체 길이를 반환
        return len(self.file_list) # 발화 개수 그대로 반환

    def __getitem__(self, idx): # 인덱스 하나에 대한 데이터를 반환하는 필수 메서드
        wav_name, start_ms, end_ms, speaker = self.file_list[idx] # idx번째 발화 정보 꺼내기
        wav_path = self.wav_path_map.get(wav_name) # 해당 wav의 전체 경로 조회

        try: # 오디오 로딩 중 에러 예외 처리
            info = sf.info(wav_path) # wav 파일의 메타 정보 조회
            native_sr = info.samplerate # 원본 파일의 실제 샘플레이트
            start_frame = int(start_ms / 1000 * native_sr) # 시작 시각을 원본 샘플레이트 기준 프레임 번호로 환산
            stop_frame = int(end_ms / 1000 * native_sr) # 끝 시각도 동일
            y, orig_sr = sf.read(wav_path, start=start_frame, stop=stop_frame, dtype='float32') # 필요 구간 읽기
            if orig_sr != self.sr: # 원본 샘플레이트가 다르면
                y = librosa.resample(y, orig_sr=orig_sr, target_sr=self.sr) # 16kHz로 리샘플링
        except Exception: # 파일 손상 등으로 읽기 실패 시
            y = np.zeros(self.max_len, dtype=np.float32) # 0으로 채운 배열로 대체 (무음)

        if y is None or len(y) == 0: # 예외 없이 빈 배열이 반환된 경우 
            y = np.zeros(self.max_len, dtype=np.float32) # 0으로 대체

        if len(y) < self.max_len: # 읽은 길이가 2.5초 보다 짧으면
            y = np.pad(y, (0, self.max_len - len(y))) # 뒤를 0으로 채움 (패딩)
        else: # 목표보다 길거나 같으면
            y = y[:self.max_len] # 2.5초에 해당하는 부분만 잘라서 사용

        y = (y - y.mean()) / (y.std() + 1e-7) # 평균 0, 표준편차 1로 정규화 (1e-7은 분모가 0이 되는 경우 방지)
        return torch.tensor(y, dtype=torch.float32), int(speaker) # 파형 텐서와 화자 라벨을 함께 반환

train_dataset = WavLMDataset(train_list, wav_path_map, sr=16000, max_duration=2.5) # train Dataset
val_dataset = WavLMDataset(val_list, wav_path_map, sr=16000, max_duration=2.5) # val Dataset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=8) # train 배치 로더
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=8) # val 배치 로더

In [6]:
# 6. 모델 정의 + 층별 학습률 구성

class AttentionPooling(nn.Module): # 시간 축 가중 평균을 위한 모듈
    def __init__(self, hidden_size): 
        super().__init__() 
        self.attention = nn.Linear(hidden_size, 1) # 프레임 벡터 -> 중요도 점수 1개로 매핑하는 선형층

    def forward(self, hidden_states): # 순전파 정의
        weights = torch.softmax(self.attention(hidden_states), dim=1) # 프레임별 점수를 시간축 기준 softmax로 정규화
        return (hidden_states * weights).sum(dim=1) # 가중치를 곱한 뒤 시간축으로 합산 -> 발화 1개 당 벡터 1개

class WavLMClassifierV2(nn.Module): # 전체 분류 모델
    def __init__(self, num_classes=2, num_layers_to_keep=12, unfreeze_last_n=12): 
        super().__init__()
        # 이 줄은 인터넷에서 WavLM 가중치를 내려받습니다. 이 노트북은 "학습 과정을 기록하는 자료"이고
        # 채점 시점에 실행되는 게 아니므로 인터넷 사용이 문제되지 않습니다.
        # (반면 제출용 추론 스크립트인 model.py는 채점 서버가 인터넷이 안 될 수 있다는 규정 때문에,
        # 여기와 다르게 로컬에 미리 저장해 둔 config만 읽도록 별도로 구성했습니다)
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base", use_safetensors=True) # WavLM 로드
        self.wavlm.feature_extractor._freeze_parameters() # 저수준 특징 추출기 가중치 고정
        self.wavlm.encoder.layers = nn.ModuleList(list(self.wavlm.encoder.layers)[:num_layers_to_keep]) # 인코더 층을 앞에서부터 num_layers_to_keep 개수만 유지
        for p in self.wavlm.encoder.parameters(): # 인코더 모든 파라미터에 대해서
            p.requires_grad = False # 전수 학습 되지 않도록 설정
        for layer in self.wavlm.encoder.layers[-unfreeze_last_n:]: # 마지막 unfreeze_last_n개 층에서 대해서만 학습
            for p in layer.parameters(): # 그 층의 파라미터
                p.requires_grad = True # 다시 학습 가능하도록 설정
        hidden_size = self.wavlm.config.hidden_size # WavLM 출력 벡터의 차원 수 조회
        self.attention_pool = AttentionPooling(hidden_size) # Attention Pooling 층 생성
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(hidden_size, num_classes)) # Dropout + 선형층으로 구성된 분류기

    def forward(self, input_values): # 순전파 정의
        h = self.wavlm(input_values).last_hidden_state # 파형을 WavLM에 통과시켜 프레임별 벡터 획득
        return self.classifier(self.attention_pool(h)) # Attention Pooling으로 요약 후 분류기에 통과시켜 결과 반환

UNFREEZE_LAYERS = 12 # 학습 대상으로 삼을 층 수 (3 ~ 12층 순차적으로 작업 수행)
BASE_LR = 1e-5 # 기준 학습률
LR_DECAY = 0.9   # 아래쪽 층일수록 곱해지는 감쇠 비율

def build_training(): # 모델 + optimizer + scheduler를 새로 구성하는 함수
    model = WavLMClassifierV2(num_classes=2, num_layers_to_keep=12, 
                              unfreeze_last_n=UNFREEZE_LAYERS).to(device) # 모델 인스턴스 생성 후 GPU로 이동
    layers = model.wavlm.encoder.layers # 인코더 층 리스트 참조
    n = len(layers) # 층 개수 확인
    groups, used = [], set() # optimizer에 넘길 파라미터 그룹 리스트 (중복 방지용 집합)
    
    for i, layer in enumerate(layers): # 층 마다 인덱스와 함께 순회
        ps = [p for p in layer.parameters() if p.requires_grad] # 그 층에서 학습 가능한 파라미터만 추출
        groups.append({'params': ps, 'lr': BASE_LR * LR_DECAY ** (n - 1 - i)}) # 층 위치에 따라 다른 lr을 부여해 그룹 추가
        used.update(id(p) for p in ps) # 이미 그룹에 포함된 파라미터로 기록
        
    head = list(model.attention_pool.parameters()) + list(model.classifier.parameters()) # Pooling + 분류기 파라미터 모음
    groups.append({'params': head, 'lr': BASE_LR}) # 기본 학습률로 그룹 추가
    used.update(id(p) for p in head) # 포함 기록
    
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in used] # 위 그룹에 안 들어간 나머지 학습 대상 파라미터
    if rest:  # 남는게 있는 경우
        groups.append({'params': rest, 'lr': BASE_LR * LR_DECAY ** n}) # 가장 낮은 lr 그룹으로 추가
    optimizer = optim.Adam(groups, weight_decay=1e-4) # 그룹 별 lr이 적용된 Adam 생성 (weight_dacay로 과적합 억제)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1) # Val Acc가 1epoch 연속 정체되며 lr을 절반으로 조정
    return model, optimizer, scheduler # 반환 

criterion = nn.CrossEntropyLoss() # 대원 / 신고자 2클래스 분류용 손실 함수

model, optimizer, scheduler = build_training() # 모델 / optimizer / scheduler 실제 생성
total = sum(p.numel() for p in model.parameters()) # 전체 파라미터 개수 합산
train_n = sum(p.numel() for p in model.parameters() if p.requires_grad) # 학습 가능한 파라미터 개수만 합산
print(f"전체: {total:,}, 학습대상: {train_n:,} ({train_n/total*100:.2f}%)") # 전체 대비 학습 대상 비율 출력

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

전체: 94,384,243, 학습대상: 85,462,771 (90.55%)


In [45]:
# 7. 속도 및 메모리 테스트 
import gc # 메모리 정리용 모듈 (가비지 컬렉터)
 
torch.cuda.empty_cache() # GPU에 남아있는 미사용 캐시 메모리 비우기
torch.cuda.reset_peak_memory_stats() # GPU 최대 메모리 사용량 측정값 초기화
start = time.time() # 시작 시각 기록
model.train() # 모델을 학습 모드로 전환
for i, (x, y) in enumerate(tqdm(train_loader, total=50, desc="속도 테스트")): # 진행률 표시 (50개 기준)
    x, y = x.to(device), y.to(device) # 입력과 정답을 device로 이동
    loss = criterion(model(x), y) # 모델 예측과 정답을 비교해 손실 계산
    loss.backward() # 역전파로 그래디언트 계산
    optimizer.step() # 그래디언트로 파라미터 업데이트
    optimizer.zero_grad() # 다음 배치를 위해 그래디언트 초기화
    if i >= 49: # 50번째 배치까지 돌았으면 중단
        break
elapsed = time.time() - start # 50배치 처리에 걸린 시간 계산

print(f"최대 GPU 메모리: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB / 10GB") # 최대로 쓴 GPU 메모리 확인
print(f"배치당 {elapsed/50:.2f}초 → 1 epoch 예상 {len(train_list)//32 * elapsed/50/60:.1f}분") # 배치당 평균 시간으로 1epoch 예상 시간 계산

del model, optimizer, scheduler # 테스트로 흔들린 객체들 삭제
gc.collect() # 파이썬 가비지 컬렉션 수행
torch.cuda.empty_cache()  # GPU 캐시 비우기

속도 테스트:  98%|████████████▋| 49/50 [00:09<00:00,  4.99it/s]


최대 GPU 메모리: 4.86 GB / 10GB
배치당 0.20초 → 1 epoch 예상 71.9분


In [46]:
# 8. 학습 루프
model, optimizer, scheduler = build_training()   # 테스트에서 흔들린 가중치는 버리고 새로 시작

NUM_EPOCHS = 15 # 최대 epoch
PATIENCE = 4 # 성능 정체가 4번 반복되면 중지
RESUME = False   # 중단된 학습을 이어서 할지 여부

save_path = os.path.expanduser("~/mission2_wavlm/best_model_wavlm_mission2_v9_12layers.pt") # 최고 성능 모델 저장 경로
last_path = os.path.expanduser("~/mission2_wavlm/ckpt_v9_12layers_last.pt") # 이어하기 용도 체크포인트 저장

if os.path.exists(save_path) and not RESUME: # 저장 경로에 파일이 이미 있는데 새로 시작하는 경우
    raise FileExistsError(f"{save_path} 가 이미 있습니다. 덮어쓰기 방지를 위해 중단합니다.") # 덮어쓰지 않도록 처리

best_val_loss, counter, start_epoch = float('inf'), 0, 0 # 최고 Val loss, 정체 카운터, 시작 epoch 초기값 설정
if RESUME and os.path.exists(last_path): # 이어하기 시도 + 체크 포인트가 있는 경우
    ck = torch.load(last_path, map_location=device, weights_only=False)   # 해당 파일 로드
    model.load_state_dict(ck['model']) # 모델 가중치 복원
    optimizer.load_state_dict(ck['optimizer']) # optimizer 상태 복원
    scheduler.load_state_dict(ck['scheduler']) # scheduler 상태 복원 
    start_epoch, best_val_loss, counter = ck['epoch'] + 1, ck['best_val_loss'], ck['counter'] # 중단된 다음 epoch부터 재개하도록 값 복원
    print(f"Epoch {start_epoch + 1}부터 이어서 학습")

epoch_times, total_start = [], time.time() # epoch별 소요 시간 기록용 리스트, 전체 시작 시각

for epoch in range(start_epoch, NUM_EPOCHS): # start_epoch부터 NUM_EPOCH 직전까지 반복
    epoch_start = time.time() # 이번 epoch 시작 시각 기록

    model.train() # 학습 모드 전환
    train_loss, train_correct = 0, 0 # 이번 epoch의 누적 손실 / 정답 수 초기화
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]") # train 진행률 표시줄 생성
    for x, y in bar: # train 배치를 하나씩 꺼내며 반복
        x, y = x.to(device), y.to(device) # 입력 / 정답을 device로 이동
        optimizer.zero_grad() # 이전 그래디언트 초기화
        outputs = model(x) # 모델에 입력을 넣어 예측값 획득
        loss = criterion(outputs, y) # 예측, 정답으로 손실 계산
        loss.backward() # 역전파로 그래디언트 계산
        optimizer.step() # 파라미터 업데이트
        train_loss += loss.item() * x.size(0) # 배치 손실을 배치 크기만큼 가중해 누적
        train_correct += (outputs.argmax(1) == y).sum().item() # 이번 배치에서 맞춘 개수 누적
        bar.set_postfix(loss=loss.item()) # 진행률 표시줄에 현재 배치 loss 표시
    train_loss /= len(train_dataset) # 누적 손실을 전체 train 샘플 수로 나누어 평균 손실 계산
    train_acc = train_correct / len(train_dataset) # 누적 정답 수를 전체 샘플 수로 나누어 정확도 계산

    model.eval() # 평가 모드 전환
    val_loss, val_correct = 0, 0 # val 누적 손실 / 정답 수 초기화
    with torch.no_grad(): # 평가 중 그래디언트 계산이 불필요하기 때문에 꺼서 메모리 절약
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"): # val 배치를 하나씩 꺼내서 반복
            x, y = x.to(device), y.to(device) # device로 이동
            outputs = model(x) # 예측값 획득
            val_loss += criterion(outputs, y).item() * x.size(0) # 배치 손실 누적
            val_correct += (outputs.argmax(1) == y).sum().item() # 맞춘 개수 누적
    val_loss /= len(val_dataset) # 평균 val loss 계산
    val_acc = val_correct / len(val_dataset) # 평균 val acc 계산

    scheduler.step(val_acc) # scheduler에 이번 epoch의 val_acc를 알려 lr 감소 여부 판단
    lr_top = optimizer.param_groups[len(model.wavlm.encoder.layers) - 1]['lr']   # 마지막 층 lr 조회

    epoch_times.append(time.time() - epoch_start) # 이번 epoch 소요 시간을 리스트에 추가
    remain = (sum(epoch_times) / len(epoch_times)) * (NUM_EPOCHS - epoch - 1) # 평균 epoch 시간 * 남은 epoch 수로 예상 잔여시간 출력
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | " # 학습 지표 출력
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | LR(top): {lr_top:.2e}") # 검증 지표 및 현재 lr 출력
    print(f"  ⏱ 이번 epoch: {epoch_times[-1]/60:.1f}분 | 누적: {(time.time()-total_start)/60:.1f}분 | 남은 예상: {remain/60:.1f}분") # 시간 정보

    stop = False # 조기 종료 여부 플래그
    if val_loss < best_val_loss: # 이번 val_loss가 지금까지 중 가장 낮으면
        best_val_loss, counter = val_loss, 0 # 최고 기록 갱신, 정체 카운터 초기화
        torch.save(model.state_dict(), save_path) # 현재 모델 가중치를 최고 성능 파일로 저장
        print("  → 최고 성능 갱신, 모델 저장") # 갱신 알림 
    else: # 개선되지 않으면
        counter += 1 # 정체 카운터 1 증가
        if counter >= PATIENCE: # 카운터가 patience 이상이면
            print(f"Early stopping (patience={PATIENCE} 초과)")
            stop = True # 종료

    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), # 이어하기용으로 모델 / optimizer 상태 저장
                'scheduler': scheduler.state_dict(), 'epoch': epoch, # 스케줄러 상태와 현재 epoch 번호 저장
                'best_val_loss': best_val_loss, 'counter': counter}, last_path) # 최고 기록과 카운터도 함께 저장
    if stop: # 종료 플래그에 따라 중지
        break
    print()

print(f"\n전체 학습 완료 | 총 소요 시간: {(time.time()-total_start)/60:.1f}분") # 전체 학습 소요 시간
print("[참고] 7층 최고 기록: Val Acc 92.01% (Epoch 14)") # 이전 실험과 비교를 위한 참고용 기록 출력

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Epoch 1/15 [Train]: 100%|█| 21982/21982 [1:13:03<00:00,  5.02it
Epoch 1/15 [Val]: 100%|████| 5525/5525 [07:50<00:00, 11.74it/s]


Epoch 1/15 | Train Loss: 0.2285, Train Acc: 0.8926 | Val Loss: 0.2047, Val Acc: 0.9089 | LR(top): 1.00e-05
  ⏱ 이번 epoch: 80.9분 | 누적: 80.9분 | 남은 예상: 1132.6분
  → 최고 성능 갱신, 모델 저장



Epoch 2/15 [Train]: 100%|█| 21982/21982 [1:12:58<00:00,  5.02it
Epoch 2/15 [Val]: 100%|████| 5525/5525 [07:51<00:00, 11.71it/s]


Epoch 2/15 | Train Loss: 0.1856, Train Acc: 0.9119 | Val Loss: 0.1790, Val Acc: 0.9172 | LR(top): 1.00e-05
  ⏱ 이번 epoch: 80.8분 | 누적: 161.7분 | 남은 예상: 1051.3분
  → 최고 성능 갱신, 모델 저장



...

Epoch 13/15 | Train Loss: 0.1511, Train Acc: 0.9266 | Val Loss: 0.1543, Val Acc: 0.9259 | LR(top): 5.00e-06
  ⏱ 이번 epoch: 80.9분 | 누적: 1052.9분 | 남은 예상: 161.9분



Epoch 14/15 [Train]: 100%|█| 21982/21982 [1:12:56<00:00,  5.02i
Epoch 14/15 [Val]: 100%|███| 5525/5525 [07:47<00:00, 11.83it/s]


Epoch 14/15 | Train Loss: 0.1465, Train Acc: 0.9288 | Val Loss: 0.1463, Val Acc: 0.9300 | LR(top): 5.00e-06
  ⏱ 이번 epoch: 80.7분 | 누적: 1133.6분 | 남은 예상: 81.0분
  → 최고 성능 갱신, 모델 저장



Epoch 15/15 [Train]: 100%|█| 21982/21982 [1:13:00<00:00,  5.02i
Epoch 15/15 [Val]: 100%|███| 5525/5525 [07:50<00:00, 11.75it/s]


Epoch 15/15 | Train Loss: 0.1462, Train Acc: 0.9288 | Val Loss: 0.1467, Val Acc: 0.9287 | LR(top): 5.00e-06
  ⏱ 이번 epoch: 80.8분 | 누적: 1214.5분 | 남은 예상: 0.0분


전체 학습 완료 | 총 소요 시간: 1214.5분
[참고] 7층 최고 기록: Val Acc 92.01% (Epoch 14)


In [7]:
# 9. 최종 채택 모델 불러오기 및 검증 데이터 평가

model_12layer = WavLMClassifierV2(num_classes=2, num_layers_to_keep=12, unfreeze_last_n=12).to(device) # 12층 구조로 모델 생성 후 device로 이동
model_12layer.load_state_dict(torch.load( # 저장된 체크 포인트 파일 불러오기
    os.path.expanduser("~/mission2_wavlm/best_model_wavlm_mission2_v9_12layers.pt"),  # 최고 성능으로 저장된 파일 경로
    map_location=device, weights_only=True)) # device에 맞게 가중치만 로드
model_12layer.eval() # 평가 모드
print("12층 모델 불러오기 완료 (Val Acc 93.0%)") # 불러오기

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

12층 모델 불러오기 완료 (Val Acc 93.0%)


In [8]:
# 10. 평가 전용 dataloader
val_loader_light = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

In [9]:
# 11. 검증 데이터 전체에 대해 예측 확률 출력
def get_probs(model, loader): # 모델과 로더를 받아 예측 확률/정답 반환
    model.eval() # 평가 모드
    all_probs, all_labels = [], [] # 전체 배치 확률/정답 리스트
    with torch.no_grad(): # 그래디언트 계산 비활성화
        for x, y in tqdm(loader, desc="예측 중"): 
            x, y = x.to(device), y.to(device) 
            outputs = model(x) # 모델 예측 획득
            probs = torch.softmax(outputs, dim=1) # 해당 예측을 확률로 변환
            all_probs.append(probs.cpu().numpy()) # 확률을 numpy로 변환 후 리스트에 추가
            all_labels.append(y.cpu().numpy()) # 정답도 동일하게 수행
    return np.concatenate(all_probs), np.concatenate(all_labels) # 모든 배치를 하나로 붙여 반환

probs_12layer, labels = get_probs(model_12layer, val_loader_light) # val 전체에 대한 확률/정답 출력

예측 중: 100%|███████████████████████████████████████████████████████████████████| 11049/11049 [09:18<00:00, 19.79it/s]


In [20]:
# 12. 성능 지표 확인

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score # 평가 지표 계산용 함수

preds_12layer = probs_12layer.argmax(1) # 확률이 더 높은 클래스를 최종 예측으로 선택
acc = accuracy_score(labels, preds_12layer) # 전체 정확도 계산
cm = confusion_matrix(labels, preds_12layer) # confusion matrix 계산

print(f"12층 모델 Val Accuracy: {acc:.4f}") # 전체 정확도 출력
print(cm) # confusion matrix 출력
print()
print(classification_report(labels, preds_12layer, target_names=['대원(0)', '신고자(1)']))
# 클래스별 precision / recall / f1 출력

12층 모델 Val Accuracy: 0.9300
[[74709  9962]
 [ 2414 89699]]

              precision    recall  f1-score   support

       대원(0)       0.97      0.88      0.92     84671
      신고자(1)       0.90      0.97      0.94     92113

    accuracy                           0.93    176784
   macro avg       0.93      0.93      0.93    176784
weighted avg       0.93      0.93      0.93    176784



In [22]:
# 13. 발화 길이 구간별 정확도

durations = np.array([(end - start) / 1000 for _, start, end, _ in val_list])

bins = [0, 1.0, 2.5, 4.2, 100] # 길이 구간 경계값 설정
for i in range(len(bins)-1): # 구간 개수만큼 반복
    mask = (durations >= bins[i]) & (durations < bins[i+1]) # 해당 구간 샘플만
    if mask.sum() > 0: # 그 구간 샘플이 있다면
        acc_bin = (preds_12layer[mask] == labels[mask]).mean() # 해당 구간 샘플들의 정확도 계산
        print(f"{bins[i]}~{bins[i+1]}초: 샘플 {mask.sum()}개, 정확도 {acc_bin:.4f}") # 구간, 샘플 수, 정확도 출력

0~1.0초: 샘플 55787개, 정확도 0.9039
1.0~2.5초: 샘플 75022개, 정확도 0.9226
2.5~4.2초: 샘플 28345개, 정확도 0.9690
4.2~100초: 샘플 17630개, 정확도 0.9813


## 14. 공식 Validation(3,640통화) 최종 평가

학습과 체크포인트 선택에 쓰이지 않은 대회 공식 Validation으로 최종 모델을 평가합니다.
- 발화 필터링은 학습과 같은 기준(300ms 미만 제외)을 적용합니다.
- 전사 텍스트(`text`)는 **모델 입력에 쓰지 않고**, 평가 후 '[개인정보]' 비식별화 구간 포함 여부를 나눠서 보는 사후 분석에만 사용합니다.
- 아래 경로를 공식 Validation 위치로 맞춘 뒤 실행하세요.

In [10]:
# 14. 공식 Validation 평가 (모델 입력은 음성 구간만, text는 사후 분석용)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score # 평가 지표 계산용 함수

OFFICIAL_LABEL_DIR = os.path.expanduser("~/mission2_wavlm/data/validation/labels")
OFFICIAL_AUDIO_DIR = os.path.expanduser("~/mission2_wavlm/data/validation/wav")

off_wav_map = {os.path.basename(p): p for p in glob.glob(os.path.join(OFFICIAL_AUDIO_DIR, "**", "*.wav"), recursive=True)} # 파일명 -> 경로
off_list, off_masked = [], [] # 평가 대상 발화 목록, '[개인정보]' 포함 여부(사후 분석용)

for jp in sorted(glob.glob(os.path.join(OFFICIAL_LABEL_DIR, "**", "*.json"), recursive=True)): # 모든 json 순회
    wav_name = os.path.basename(jp).replace(".json", ".wav") # 대응 wav 파일명
    if wav_name not in off_wav_map: # wav가 없으면 건너뜀
        continue
    with open(jp, "r", encoding="utf-8") as f: # json 열기
        data = json.load(f)
    for utt in data.get("utterances", []): # 통화 안의 모든 발화
        s, e, spk = utt.get("startAt"), utt.get("endAt"), utt.get("speaker")
        if s is None or e is None or e <= s or spk is None or (e - s) < 300: # 비정상이거나 300ms 미만이면 제외 (학습과 동일 기준)
            continue
        off_list.append((wav_name, s, e, spk)) # 평가 대상에 추가
        off_masked.append("[개인정보]" in (utt.get("text") or "")) # 사후 분석용 플래그 (모델 입력에는 사용하지 않음)

off_masked = np.array(off_masked) # 불리언 배열로 변환
off_loader = DataLoader(WavLMDataset(off_list, off_wav_map, sr=16000, max_duration=2.5), batch_size=16, shuffle=False, num_workers=0) # 평가 로더
off_probs, off_labels = get_probs(model_12layer, off_loader) # 예측 확률 / 정답 (11번 셀의 함수 재사용)
off_preds = off_probs.argmax(1) # 확률이 큰 쪽 선택 (threshold 0.5와 동일)

print(f"공식 Validation 발화 조각: {len(off_list)}개")
print(f"전체 Accuracy: {accuracy_score(off_labels, off_preds):.4f}") # 전체 정확도
print(confusion_matrix(off_labels, off_preds)) # 행=실제(대원, 신고자), 열=예측
print(classification_report(off_labels, off_preds, target_names=["대원(0)", "신고자(1)"]))

keep = ~off_masked # 비식별화 구간을 제외한 발화만
print(f"[개인정보] 포함 발화 비율: {off_masked.mean()*100:.2f}%")
print(f"비식별화 구간 제외 Accuracy: {accuracy_score(off_labels[keep], off_preds[keep]):.4f}")
print(classification_report(off_labels[keep], off_preds[keep], target_names=["대원(0)", "신고자(1)"])) # 제외 후 화자별 recall

예측 중: 100%|█████████████████████████████████████████████████████████████████████| 6922/6922 [05:33<00:00, 20.76it/s]

공식 Validation 발화 조각: 110743개
전체 Accuracy: 0.9264
[[46737  6486]
 [ 1665 55855]]
              precision    recall  f1-score   support

       대원(0)       0.97      0.88      0.92     53223
      신고자(1)       0.90      0.97      0.93     57520

    accuracy                           0.93    110743
   macro avg       0.93      0.92      0.93    110743
weighted avg       0.93      0.93      0.93    110743

[개인정보] 포함 발화 비율: 12.35%
비식별화 구간 제외 Accuracy: 0.9707
              precision    recall  f1-score   support

       대원(0)       0.97      0.97      0.97     47863
      신고자(1)       0.97      0.97      0.97     49201

    accuracy                           0.97     97064
   macro avg       0.97      0.97      0.97     97064
weighted avg       0.97      0.97      0.97     97064

